In [1]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

In [2]:
base_dir = os.path.expanduser('~/Projects/groundwater/data/henry_data/all_scenarios')
coupling_scenarios_dir = os.path.join(base_dir, 'coupling_scenarios')
diffusion_scenarios_dir = os.path.join(base_dir, 'diffusion_scenarios')
os.path.exists(coupling_scenarios_dir), os.path.exists(diffusion_scenarios_dir)

(True, True)

In [3]:
# Build scenario maps for both coupling and diffusion groups so controls stay synchronized.
def build_scenario_map(root_dir):
    scenario_map = {}
    if not os.path.exists(root_dir):
        return scenario_map

    for scenario in sorted(os.listdir(root_dir)):
        if scenario.startswith('.'):
            continue

        scenario_dir = os.path.join(root_dir, scenario)
        scenario_cfg_path = os.path.join(scenario_dir, 'scenario_config.json')
        runs_cfg_path = os.path.join(scenario_dir, 'runs_config.json')

        if not os.path.exists(scenario_cfg_path):
            continue

        with open(scenario_cfg_path, 'r') as f:
            scenario_config = json.load(f)

        runs_config = {'runs': []}
        if os.path.exists(runs_cfg_path):
            with open(runs_cfg_path, 'r') as f:
                runs_config = json.load(f)

        run_records = runs_config.get('runs', [])
        run_meta_by_name = {rec.get('run'): rec for rec in run_records if rec.get('run')}

        runs = []
        for run in sorted(os.listdir(scenario_dir)):
            if run.startswith('.') or not run.startswith('run_'):
                continue

            windows_path = os.path.join(scenario_dir, run, 'windows.npz')
            if os.path.exists(windows_path):
                runs.append(run)

        if runs:
            scenario_map[scenario] = {
                'config': scenario_config,
                'runs': runs,
                'run_meta_by_name': run_meta_by_name,
            }

    return scenario_map

group_to_root = {
    'coupling': coupling_scenarios_dir,
    'diffusion': diffusion_scenarios_dir,
}

scenario_maps = {
    'coupling': build_scenario_map(coupling_scenarios_dir),
    'diffusion': build_scenario_map(diffusion_scenarios_dir),
}

available_groups = [g for g in ['coupling', 'diffusion'] if scenario_maps[g]]
if not available_groups:
    raise ValueError('No scenarios with windows.npz were found in coupling or diffusion directories.')

group_dropdown = widgets.Dropdown(
    options=available_groups,
    value=available_groups[0],
    description='Group:',
    layout=widgets.Layout(width='55%')
)

scenario_options = list(scenario_maps[group_dropdown.value].keys())
scenario_dropdown = widgets.Dropdown(
    options=scenario_options,
    value=scenario_options[0],
    description='Scenario:',
    layout=widgets.Layout(width='55%')
)

run_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(scenario_maps[group_dropdown.value][scenario_options[0]]['runs']) - 1,
    step=1,
    description='Run index:',
    continuous_update=False,
    layout=widgets.Layout(width='55%')
)

run_label = widgets.HTML()
plot_output = widgets.Output()

def update_scenario_options(*_):
    selected_group = group_dropdown.value
    options = list(scenario_maps[selected_group].keys())
    scenario_dropdown.options = options
    scenario_dropdown.value = options[0]

def update_run_slider(*_):
    selected_group = group_dropdown.value
    runs = scenario_maps[selected_group][scenario_dropdown.value]['runs']
    run_slider.max = len(runs) - 1
    run_slider.value = 0

def render_plot(*_):
    selected_group = group_dropdown.value
    scenario = scenario_dropdown.value
    scenario_map = scenario_maps[selected_group]
    scenario_config = scenario_map[scenario]['config']
    runs = scenario_map[scenario]['runs']
    run_meta_by_name = scenario_map[scenario]['run_meta_by_name']
    run = runs[run_slider.value]
    run_meta = run_meta_by_name.get(run, {})

    beta_c = scenario_config.get('beta_c', 'N/A')
    diffc = scenario_config.get('diffc', 'N/A')
    scenario_group = scenario_config.get('scenario_group', selected_group)

    hk = run_meta.get('hk', 'N/A')
    por = run_meta.get('por', 'N/A')
    inflow = run_meta.get('inflow', 'N/A')

    windows_path = os.path.join(group_to_root[selected_group], scenario, run, 'windows.npz')
    windows_data = np.load(windows_path)
    final_window = windows_data['output_tensor'][-1]

    run_label.value = f'<b>Run:</b> {run} ({run_slider.value + 1}/{len(runs)})'

    with plot_output:
        clear_output(wait=True)
        fig, ax = plt.subplots(1, 2, figsize=(12, 3))
        im0 = ax[0].imshow(final_window[0], cmap='Reds')
        im1 = ax[1].imshow(final_window[1], cmap='Blues')
        ax[0].set_title('Final Window - Concentration')
        ax[1].set_title('Final Window - Head')
        plt.colorbar(im0, ax=ax[0])
        plt.colorbar(im1, ax=ax[1])
        plt.suptitle(
            f'Group: {scenario_group}, Scenario: {scenario}, Beta_c: {beta_c}, Diffc: {diffc}, HK: {hk}, POR: {por}, Inflow: {inflow}'
        )
        plt.tight_layout()
        plt.show()

group_dropdown.observe(update_scenario_options, names='value')
group_dropdown.observe(update_run_slider, names='value')
group_dropdown.observe(render_plot, names='value')
scenario_dropdown.observe(update_run_slider, names='value')
scenario_dropdown.observe(render_plot, names='value')
run_slider.observe(render_plot, names='value')

controls = widgets.VBox([group_dropdown, scenario_dropdown, run_slider, run_label])
display(controls, plot_output)
render_plot()

Output()

In [8]:
clean_dirty = os.path.expanduser('~/Projects/groundwater/data/henry_data/all_scenarios_clean')

# Copy all files except *.hds and *.ucn from base_dir to clean_dirty, preserving directory structure.
import shutil
def copy_scenarios_exclude_hds_ucn(src_dir, dst_dir):
    if not os.path.exists(dst_dir):
        os.makedirs(dst_dir)

    for root, dirs, files in os.walk(src_dir):
        relative_path = os.path.relpath(root, src_dir)
        dst_root = os.path.join(dst_dir, relative_path)

        if not os.path.exists(dst_root):
            os.makedirs(dst_root)

        for file in files:
            if file.endswith('.hds') or file.endswith('.ucn'):
                continue
            src_file = os.path.join(root, file)
            dst_file = os.path.join(dst_root, file)
            shutil.copy2(src_file, dst_file)

copy_scenarios_exclude_hds_ucn(base_dir, clean_dirty)
